In [1]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import f1_score
from rouge_score import rouge_scorer
import torch
import nltk
import numpy as np
# Import torch.nn for neural network modules
import torch.nn as nn
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
class NonCausalT5(T5ForConditionalGeneration):
    def __init__(self, config):
        super(NonCausalT5, self).__init__(config)
        # Add relative positional encodings
        self.relative_position_embeddings = nn.Embedding(config.n_positions, config.d_model)

    def _prepare_attention_mask(self, input_ids):
        """
        Modify the attention mask to support non-causal (full attention) across the decoder.
        """
        # In the non-causal setting, allow full attention, i.e., no masking
        attention_mask = input_ids.ne(self.config.pad_token_id).float()
        return attention_mask

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        labels=None,
        **kwargs  # Accept extra args but DO NOT pass to decoder
    ):
        # Prepare attention mask if not provided
        if attention_mask is None:
            attention_mask = self._prepare_attention_mask(input_ids)

        # Encoder part (same as T5)
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Decoder input preparation (if not provided, use input_ids as default)
        if decoder_input_ids is None:
            decoder_input_ids = input_ids

        # If no decoder attention mask, use full attention (all ones)
        if decoder_attention_mask is None:
            decoder_attention_mask = torch.ones_like(decoder_input_ids).float()

        # Add relative position encoding to decoder input embeddings
        seq_len = decoder_input_ids.size(1)
        position_ids = torch.arange(seq_len, dtype=torch.long, device=decoder_input_ids.device)
        position_embeddings = self.relative_position_embeddings(position_ids)

        # Combine token embeddings and position embeddings for the decoder
        decoder_input_embeds = self.decoder.embed_tokens(decoder_input_ids) + position_embeddings

        # Decoder forward pass (removed **kwargs from here)
        decoder_outputs = self.decoder(
            attention_mask=decoder_attention_mask,
            encoder_hidden_states=encoder_outputs.last_hidden_state,
            encoder_attention_mask=attention_mask,
            inputs_embeds=decoder_input_embeds,
            return_dict=True
        )

        # Get logits from decoder output
        logits = self.lm_head(decoder_outputs.last_hidden_state)

        # If labels are provided, compute the loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id)
            loss = loss_fct(logits.view(-1, self.config.vocab_size), labels.view(-1))

        if loss is not None:
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits}


In [3]:
# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = NonCausalT5.from_pretrained("t5-small")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Some weights of NonC

In [4]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.columns)


Index(['answers', 'query', 'finalpassage'], dtype='object')


In [5]:
# Load train.csv
df = pd.read_csv("train.csv")
df = df[:200]
df = df.dropna(subset=['query', 'answers'])
df['input_text'] = 'chatbot: ' + df['query']
df['target_text'] = df['answers']
train_dataset = Dataset.from_pandas(df[['input_text', 'target_text']])

In [6]:
# Preprocess function
def preprocess(example):
    inputs = tokenizer(example['input_text'], padding="max_length", truncation=True, max_length=64)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example['target_text'], padding="max_length", truncation=True, max_length=64)
    inputs['labels'] = labels['input_ids']
    return inputs

# Tokenize training dataset
tokenized_train = train_dataset.map(preprocess)


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [7]:
# Hyperparameters
training_args = TrainingArguments(
    output_dir="./t5_chatbot",
    per_device_train_batch_size=16,
    num_train_epochs=4,
    learning_rate=3e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=20,
    save_total_limit=1,
    save_strategy="epoch",
    evaluation_strategy="no"
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

# Train the model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: rahulravi724 (rahulravi724-loyalist-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
20,219.417000
40,147.698700


TrainOutput(global_step=52, training_loss=163.23102510892429, metrics={'train_runtime': 33.0877, 'train_samples_per_second': 24.178, 'train_steps_per_second': 1.572, 'total_flos': 13534180147200.0, 'train_loss': 163.23102510892429, 'epoch': 4.0})

In [9]:

# Save the model
model.save_pretrained("./t5_chatbot")
tokenizer.save_pretrained("./t5_chatbot")


('./t5_chatbot/tokenizer_config.json',
 './t5_chatbot/special_tokens_map.json',
 './t5_chatbot/spiece.model',
 './t5_chatbot/added_tokens.json')

In [10]:
# Load valid.csv
valid_df = pd.read_csv("valid.csv")
valid_df = valid_df.dropna(subset=['query', 'answers'])
valid_df['input_text'] = 'chatbot: ' + valid_df['query']


In [24]:
def generate_response(input_text):
    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    # Encoder forward pass (same as before)
    encoder_outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)

    # Decoder part
    # We need to pass input_ids or inputs_embeds to the decoder.
    # Add relative position embeddings to the decoder inputs
    position_ids = torch.arange(input_ids.size(1), dtype=torch.long, device=input_ids.device)
    position_embeddings = model.relative_position_embeddings(position_ids)

    # Use `inputs_embeds` for the decoder input
    decoder_input_embeds = model.decoder.embed_tokens(input_ids) + position_embeddings  # Modify input embeddings

    # Now perform the decoder's forward pass
    decoder_attention_mask = torch.ones_like(input_ids).float()  # No masking, full attention
    decoder_outputs = model.decoder(
        attention_mask=decoder_attention_mask,
        encoder_hidden_states=encoder_outputs.last_hidden_state,
        encoder_attention_mask=attention_mask,
        inputs_embeds=decoder_input_embeds  # Use modified embeddings
    )

    # Generate the logits
    logits = model.lm_head(decoder_outputs.last_hidden_state)

    # Get predicted tokens (for generation task)
    predicted_ids = torch.argmax(logits, dim=-1)

    # Decode the predicted ids to text
    predicted_answer = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

    return predicted_answer


In [25]:
valid_df['predicted_answer'] = valid_df['query'].apply(generate_response)


In [26]:
# ROUGE-L Score Evaluation
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_l_scores = [
    scorer.score(ref, pred)['rougeL'].fmeasure
    for ref, pred in zip(valid_df["answers"], valid_df["predicted_answer"])
]
avg_rouge_l = np.mean(rouge_l_scores)
print(f"\nAverage ROUGE-L Score: {avg_rouge_l:.4f}")


Average ROUGE-L Score: 0.0740


In [27]:
# BERT F1 Score Evaluation
def compute_bert_f1(true, pred):
    true_tokens = tokenizer.tokenize(true)
    pred_tokens = tokenizer.tokenize(pred)
    common_tokens = set(true_tokens) & set(pred_tokens)
    precision = len(common_tokens) / len(pred_tokens) if pred_tokens else 0
    recall = len(common_tokens) / len(true_tokens) if true_tokens else 0
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0
    return f1

bert_f1_scores = [
    compute_bert_f1(ref, pred)
    for ref, pred in zip(valid_df["answers"], valid_df["predicted_answer"])
]
avg_bert_f1 = np.mean(bert_f1_scores)
print(f"\nAverage BERT F1 Score: {avg_bert_f1:.4f}")


Average BERT F1 Score: 0.0988
